# Phase 2 — Build the Harrier offline bundle
Run with **Internet enabled**, then use **Save Version → Create Dataset from Output**. This bundle is isolated from the Phase 1 bundle and replaces only `vietlegal-e5` with approved `vietlegal-harrier-0.6b`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/UIT-LegalIR.git'
REPO_REF = 'main'  # Prefer the immutable commit containing this Phase 2 folder.
EXPERIMENT_ID = 'phase2-vietlegal-harrier-0.6b'
BUNDLE_ROOT = Path('/kaggle/working/legalir-phase2-harrier-bundle')
REPO_DIR = Path('/kaggle/working/UIT-LegalIR-phase2')

def run(*command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run('git', 'clone', REPO_URL, REPO_DIR)
run('git', 'checkout', REPO_REF, cwd=REPO_DIR)
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
for directory in ('models', 'wheels', 'configs', 'manifests', 'project', 'licenses'):
    (BUNDLE_ROOT / directory).mkdir(parents=True, exist_ok=True)
print('Experiment:', EXPERIMENT_ID)
print('Bundle output:', BUNDLE_ROOT)

In [ ]:
# Download/build only; never install into the builder kernel.
run(sys.executable, '-m', 'pip', 'download', '--no-deps', '--dest', BUNDLE_ROOT / 'wheels', '--only-binary=:all:', '-r', REPO_DIR / 'requirements-offline.txt')
run(sys.executable, '-m', 'pip', 'wheel', '--no-deps', '--wheel-dir', BUNDLE_ROOT / 'wheels', REPO_DIR)
shutil.copy2(REPO_DIR / 'requirements-offline.txt', BUNDLE_ROOT / 'requirements-offline.txt')
phase2_config = REPO_DIR / 'kaggle' / 'phase2_harrier' / 'kaggle_rtx_pro_6000.yaml'
shutil.copy2(phase2_config, BUNDLE_ROOT / 'configs' / 'kaggle_rtx_pro_6000.yaml')
print('Wheel count:', len(list((BUNDLE_ROOT / 'wheels').glob('*.whl'))))

In [ ]:
import yaml
from huggingface_hub import snapshot_download

MODELS = {
    'vietlegal_harrier': {'id': 'mainguyen9/vietlegal-harrier-0.6b', 'revision': '91a0e1ebe4b63b4475bbae40658b8ca9231bea74', 'license': 'Apache-2.0'},
    'vietnamese_embedding': {'id': 'AITeamVN/Vietnamese_Embedding_v2', 'revision': '18b44161e041bf1d3a333ab5144b5b7b93f914d2', 'license': 'Apache-2.0'},
    'nemotron': {'id': 'nvidia/Nemotron-3-Embed-1B-BF16', 'revision': 'c0c9fea93ea424587517f2c59e20db9f1d6bf615', 'license': 'OpenMDW-1.1'},
    'jina': {'id': 'jinaai/jina-reranker-v3.5', 'revision': 'e8a93f33f0b22108f8c2364f8484ce3422552fbc', 'license': 'CC-BY-NC-4.0'},
    'vietnamese_reranker': {'id': 'AITeamVN/Vietnamese_Reranker', 'revision': 'f536976248403314225d7fdfdbc87f0e9516a54e', 'license': 'Apache-2.0'},
}
config = yaml.safe_load(phase2_config.read_text(encoding='utf-8'))
assert set(config['models']) == set(MODELS), (set(config['models']), set(MODELS))
for name, spec in MODELS.items():
    assert config['models'][name]['id'] == spec['id']
    assert config['models'][name]['revision'] == spec['revision']
    destination = BUNDLE_ROOT / 'models' / name
    snapshot_download(
        repo_id=spec['id'], repo_type='model', revision=spec['revision'],
        local_dir=destination,
        ignore_patterns=['onnx/*', '*.onnx', '*.onnx_data'],
    )
    if not (destination / 'config.json').is_file():
        raise RuntimeError(f'{name} download did not contain config.json')
    if name == 'jina' and not (destination / 'modeling.py').is_file():
        raise RuntimeError('Jina custom modeling.py is required for offline loading')
    metadata_cache = destination / '.cache'
    if metadata_cache.exists():
        shutil.rmtree(metadata_cache)
    for candidate in ('LICENSE', 'LICENSE.md', 'LICENSE.txt'):
        source = destination / candidate
        if source.is_file():
            shutil.copy2(source, BUNDLE_ROOT / 'licenses' / f'{name}_{candidate}')
print('Downloaded Phase 2 snapshots:', ', '.join(MODELS))

In [ ]:
import hashlib
import json

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def records(root):
    return [
        {'path': str(path.relative_to(BUNDLE_ROOT)), 'bytes': path.stat().st_size, 'sha256': sha256(path)}
        for path in sorted(root.rglob('*')) if path.is_file()
    ]

project_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {
    'schema_version': 2,
    'experiment_id': EXPERIMENT_ID,
    'replaces': {'vietlegal_e5': 'vietlegal_harrier'},
    'project_commit': project_commit,
    'python_version': sys.version,
    'models': [{**spec, 'name': name, 'local_path': f'models/{name}'} for name, spec in MODELS.items()],
    'files': records(BUNDLE_ROOT / 'models') + records(BUNDLE_ROOT / 'wheels') + records(BUNDLE_ROOT / 'configs'),
}
(BUNDLE_ROOT / 'manifests' / 'bundle_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(BUNDLE_ROOT / 'manifests' / 'checksums.sha256').write_text(''.join(f"{item['sha256']}  {item['path']}\n" for item in manifest['files']), encoding='utf-8')
(BUNDLE_ROOT / 'project' / 'source_commit.txt').write_text(project_commit + '\n', encoding='utf-8')
print(json.dumps({'experiment': EXPERIMENT_ID, 'project_commit': project_commit, 'files': len(manifest['files'])}, indent=2))

In [ ]:
required_wheels = ('faiss_cpu-', 'sentence_transformers-', 'transformers-', 'tokenizers-', 'safetensors-', 'uit_legalir-')
wheel_names = [path.name.lower() for path in (BUNDLE_ROOT / 'wheels').glob('*.whl')]
missing_wheels = [prefix for prefix in required_wheels if not any(name.startswith(prefix) for name in wheel_names)]
if missing_wheels:
    raise RuntimeError(f'Missing wheels: {missing_wheels}')
for name in MODELS:
    model_dir = BUNDLE_ROOT / 'models' / name
    weight_files = list(model_dir.rglob('*.safetensors')) + list(model_dir.rglob('pytorch_model*.bin'))
    if not weight_files:
        raise RuntimeError(f'{name} has no PyTorch weight file')
assert 'vietlegal_e5' not in MODELS
assert 'vietlegal_harrier' in MODELS
print('Phase 2 Harrier bundle checks passed. Create a Kaggle Dataset from the bundle output.')